<a href="https://colab.research.google.com/github/ramboorgadda/ANN/blob/main/Agent_Memory_Redis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
%pip install langchain-openai langgraph-checkpoint langgraph langgraph-checkpoint-redis langchain-redis

In [32]:
import getpass
import os
from google.colab import userdata
from huggingface_hub import login
from dotenv import load_dotenv
def _set_env(key: str):
  if key not in os.environ:
    os.environ[key] = getpass.getpass(f"{key}:")

_set_env("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"open ai api key exists and starts with {openai_api_key[:8]}")
else:
    print(f"open ai api key is not found")

open ai api key exists and starts with sk-proj-


In [33]:
%%sh
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install redis-stack-server  > /dev/null 2>&1
redis-stack-server --daemonize yes

deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb jammy main
Starting redis-stack-server, database path /var/lib/redis-stack


gpg: cannot open '/dev/tty': No such device or address
curl: (23) Failed writing body


In [36]:
import os
from redis import Redis

REDIS_URL = os.getenv("REDIS_URL","redis://localhost:6379")

redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()

True

#Prepare Memory DataModel

In [57]:
from ast import Str
import ulid
from datetime import datetime
from enum import Enum
from typing import List,Optional
from pydantic import BaseModel,Field


class MemoryType(str,Enum):
  """
    Defines the type of long-term memory for categorization and retrieval.

    EPISODIC: Personal experiences and user-specific preferences
              (e.g., "User prefers Delta airlines", "User visited Paris last year")

    SEMANTIC: General domain knowledge and facts
              (e.g., "Singapore requires passport", "Tokyo has excellent public transit")

    The type of a long-term memory.

    EPISODIC: User specific experiences and preferences

    SEMANTIC: General knowledge on top of the user's preferences and LLM's
    training data.
    """

  EPISODIC = "episodic"
  SEMANTIC = "semantic"

class Memory(BaseModel):
  """Represents Single long-term memory"""
  content: str
  memory_type: MemoryType
  metadata: str

class memories(BaseModel):
  """
    A list of memories extracted from a conversation by an LLM.

    NOTE: OpenAI's structured output requires us to wrap the list in an object.
  """
  memories: List[Memory]

class StoredMemory(BaseModel):
  """A Stored long-term memory"""
  id: str
  memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
  created_at: datetime = Field(default_factory=datetime.now)
  user_id: Optional[str] = None
  thread_id: Optional[str] = None
  memory_type: Optional[MemoryType] = None
  content: str # Added content field
  metadata: str # Added metadata field

In [7]:
# Removed incorrect pip install

In [38]:
from redisvl.schema.schema import IndexSchema
from redisvl.index import SearchIndex

memory_schema  = IndexSchema.from_dict({
    "index": {
        "name": "agent_memories",
        "prefix": "memory",
        "key_separator": ":",
        "storage_type": "json"
    },
    "fields": [
        {"name": "content", "type": "text"},
            {"name": "memory_type", "type": "tag"},
            {"name": "metadata", "type": "text"},
            {"name": "created_at", "type": "text"},
            {"name": "user_id", "type": "tag"},
            {"name": "memory_id", "type": "tag"},
            {
                "name": "embedding",
                "type": "vector",
                "attrs": {
                    "algorithm": "flat",
                    "dims": 1536,  # OpenAI embedding dimension
                    "distance_metric": "cosine",
                    "datatype": "float32",
                },
            },
        ],
})

In [39]:
try:
    long_term_memory_index = SearchIndex(memory_schema, redis_client,validate_on_load=True)
    long_term_memory_index.create(overwrite=True)
    print(f"long-term memory index is ready")
except Exception as e:
    print(f"long-term memory index is already ready")

long-term memory index is ready


In [40]:
!rvl index info -i agent_memories



Index Information:
╭────────────────┬────────────────┬────────────────┬────────────────┬────────────────┬╮
│ Index Name     │ Storage Type   │ Prefixes       │ Index Options  │ Indexing       │
├────────────────┼────────────────┼────────────────┼────────────────┼────────────────┼┤
| agent_memories | JSON           | ['memory']     | []             | 0              |
╰────────────────┴────────────────┴────────────────┴────────────────┴────────────────┴╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬╮
│ Name            │ Attribute       │ Type            │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │
├─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────

In [41]:
from redisvl.utils.vectorize.text.openai import OpenAITextVectorizer
openai_embed  = OpenAITextVectorizer(model="text-embedding-ada-002")

In [42]:
import logging
logger = logging.getLogger(__name__)

In [43]:
from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag
SYSTEM_USER_ID= "system"
def similar_memory_exists(content:str,
                          memory_type: MemoryType,
                          user_id: str = SYSTEM_USER_ID,
                          thread_id: Optional[str] = None,
                          distance_threshold: float = 0.1
                          ) ->bool:

  """Check if similar longo-term memory exists"""
  content_embedding = openai_embed.embed(content)

  filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type)

  if thread_id:
    filters &= Tag("thread_id") == thread_id

  vector_query = VectorRangeQuery(
        vector=content_embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )

  results  = long_term_memory_index.query(vector_query)

  logger.debug(f"Similar memory search results: {results}")

  if results:
    logger.debug(
            f"{len(results)} similar {'memory' if results.count == 1 else 'memories'} found. First: "
            f"{results[0]['id']}. Skipping storage."
        )
    return True

  return False



In [44]:
from datetime import datetime
from typing import List, Optional, Union

import ulid


def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
):
    """Store a long-term memory in Redis with deduplication.

        This function:
        1. Checks for similar existing memories to avoid duplicates
        2. Generates vector embeddings for semantic search
        3. Stores the memory with metadata for retrieval
        """
    if metadata is None:
        metadata = "{}"

    logger.info(f"Preparing to store memory: {content}")

    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info("Similar memory found, skipping storage")
        return

    embedding = openai_embed.embed(content)

    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id,
    }

    try:
        long_term_memory_index.load([memory_data])
    except Exception as e:
        logger.error(f"Error storing memory: {e}")
        return

    logger.info(f"Stored {memory_type} memory: {content}")

In [46]:
def retrieve_memories(
    query: str,
    memory_type: Union[Optional[MemoryType], List[MemoryType]] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
    limit: int = 5,
) -> List[StoredMemory]:
    """Retrieve relevant memories from Redis using vector similarity search.

    """
    # Create vector query using query embedding
    logger.debug(f"Retrieving memories for query: {query}")
    vector_query = VectorRangeQuery(
        vector=openai_embed.embed(query),
        return_fields=[
            "content",
            "memory_type",
            "metadata",
            "created_at",
            "memory_id",
            "thread_id",
            "user_id",
        ],
        num_results=limit,
        vector_field_name="embedding",
        dialect=2,
        distance_threshold=distance_threshold,
    )

    # Build filter conditions
    base_filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]

    if memory_type:
        if isinstance(memory_type, list):
            base_filters.append(f"@memory_type:{{{'|'.join(memory_type)}}}")
        else:
            base_filters.append(f"@memory_type:{{{memory_type.value}}}")

    if thread_id:
        base_filters.append(f"@thread_id:{{{thread_id}}}")

    vector_query.set_filter(" ".join(base_filters))

    # Execute vector similarity search
    results = long_term_memory_index.query(vector_query)

    # Parse results into StoredMemory objects
    memories = []
    for doc in results:
        try:
            memory = StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id", None),
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            )
            memories.append(memory)
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
            continue
    return memories

In [54]:
from typing import Dict, Optional

from langchain_core.tools import tool
from langchain_core.runnables.config import RunnableConfig


@tool
def store_memory_tool(
    content: str,
    memory_type: MemoryType,
    metadata: Optional[Dict[str, str]] = None,
    config: Optional[RunnableConfig] = None,
) -> str:
    """
    Store a long-term memory in the system.

    Use this tool to save important information about user preferences,
    experiences, or general knowledge that might be useful in future
    interactions.
    """
    config = config or RunnableConfig()
    user_id = config.get("user_id", SYSTEM_USER_ID)
    thread_id = config.get("thread_id")

    try:
        # Store in long-term memory
        store_memory(
            content=content,
            memory_type=memory_type,
            user_id=user_id,
            thread_id=thread_id,
            metadata=str(metadata) if metadata else None,
        )

        return f"Successfully stored {memory_type} memory: {content}"
    except Exception as e:
        return f"Error storing memory: {str(e)}"

In [48]:
store_memory_tool.invoke({"content": "I like flying on Delta when possible", "memory_type": "episodic"})

'Successfully stored MemoryType.EPISODIC memory: I like flying on Delta when possible'

In [49]:
@tool
def retrieve_memories_tool(
    query: str,
    memory_type: List[MemoryType],
    limit: int = 5,
    config: Optional[RunnableConfig] = None,
) -> str:
    """
    Retrieve long-term memories relevant to the query.

    Use this tool to access previously stored information about user
    preferences, experiences, or general knowledge.
    """
    config = config or RunnableConfig()
    user_id = config.get("user_id", SYSTEM_USER_ID)

    try:
        # Get long-term memories
        stored_memories = retrieve_memories(
            query=query,
            memory_type=memory_type,
            user_id=user_id,
            limit=limit,
            distance_threshold=0.3,
        )

        # Format the response
        response = []

        if stored_memories:
            response.append("Long-term memories:")
            for memory in stored_memories:
                response.append(f"- [{memory.memory_type}] {memory.content}")

        return "\n".join(response) if response else "No relevant memories found."

    except Exception as e:
        return f"Error retrieving memories: {str(e)}"

In [66]:
retrieve_memories_tool.invoke({"query": "Italian food", "memory_type": ["episodic"]})

'Long-term memories:\n- [MemoryType.EPISODIC] I enjoy Italian food, especially pasta dishes\n- [MemoryType.EPISODIC] I like flying on Delta when possible'

In [64]:
store_memory_tool.invoke({"content": "I enjoy Italian food, especially pasta dishes", "memory_type": "episodic"})

'Successfully stored MemoryType.EPISODIC memory: I enjoy Italian food, especially pasta dishes'

# Setting up React Agent

In [67]:
from langchain_core.messages import AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt.chat_agent_executor import create_react_agent
from langgraph.checkpoint.redis import RedisSaver
redis_saver = RedisSaver(redis_client = redis_client)
redis_saver.setup()

In [70]:
tools = [store_memory_tool, retrieve_memories_tool]
llm=ChatOpenAI(model="gpt-4o",temperature = 0.7).bind_tools(tools)

travel_agent = create_react_agent(model=llm,
                                tools=tools,
                                checkpointer=redis_saver,
                                prompt=SystemMessage(
                                  content="""
                                  You are a travel assistant helping users plan their trips. You remember user preferences
                                  and provide personalized recommendations based on past interactions.

                                  You have access to the following types of memory:
                                  1. Short-term memory: The current conversation thread
                                  2. Long-term memory:
                                    - Episodic: User preferences and past trip experiences (e.g., "User prefers window seats")
                                    - Semantic: General knowledge about travel destinations and requirements

                                  Your procedural knowledge (how to search, book flights, etc.) is built into your tools and prompts.

                                  Always be helpful, personal, and context-aware in your responses.
                                  """
                                ),
)

/tmp/ipykernel_3601/667971635.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_agent = create_react_agent(model=llm,
